# PINNs for the viscous Burgers equation: the forward problem

*PINNs course · Nassim RAZAALY*

We solve the viscous Burgers equation with a physics-informed neural network (PINN), step by step:

1. **The problem**: the equation, its exact solution (Cole–Hopf), data generation, the training points.
2. **Derivatives**: how automatic differentiation gives $u_x$, $u_t$, $u_{xx}$, first on the simplest possible model.
3. **The PINN**: the network, the loss, the training (Adam, then L-BFGS), and tools to evaluate a trained model.
4. **Exercises: how to do better?** Reliability, the optimizer, the points, the network, the loss. Solutions in `c_correction.ipynb`.

The notebook is self-contained: it only needs `numpy`, `torch` and `matplotlib`, and runs in about 4 minutes on a laptop CPU.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

DTYPE = torch.float32                      # precision of the whole notebook (float32 is the PyTorch default)
torch.set_default_dtype(DTYPE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def to_torch(a):
    """numpy array -> torch tensor with the notebook precision, on the notebook device."""
    return torch.as_tensor(np.asarray(a), dtype=torch.get_default_dtype(), device=DEVICE)


def to_numpy(t):
    return t.detach().cpu().numpy()


def sci(v, d=1):
    """3.2e-04 -> $3.2\\cdot10^{-4}$ (used to write the numbers in the comments of this notebook)"""
    m, e = f'{v:.{d}e}'.split('e')
    return f'${m}$' if int(e) == 0 else f'${m}\\cdot10^{{{int(e)}}}$'


plt.rcParams.update({'figure.dpi': 100, 'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE, '| precision:', DTYPE, '| torch', torch.__version__)

## 1. The problem

### 1.1 The viscous Burgers equation

$$\partial_t u + u\,\partial_x u = \nu\,\partial_{xx} u,\qquad x\in[-1,1],\ t\in[0,T],$$
$$u(x,0) = u_0(x) = -\sin(\pi x),\qquad u(-1,t) = u(1,t) = 0,\qquad \nu = \frac{0.01}{\pi}\approx 3.2\cdot10^{-3},\qquad T = 0.99.$$
This is the benchmark of Raissi *et al.* (J. Comput. Phys., 2019).

* The term $u\,\partial_x u$ is a **nonlinear advection**: each value of $u$ travels at the speed $u$. Here $u_0>0$ for $x<0$ and $u_0<0$ for $x>0$: both sides move towards $x=0$ and the profile steepens. Without viscosity, the characteristics $x = x_0 + u_0(x_0)\,t$ first cross at $t^* = 1/\max(-u_0') = 1/\pi\approx0.32$: a shock forms.
* The term $\nu\,\partial_{xx}u$ is a **diffusion**, which stops the steepening: instead of a shock, a smooth **front** forms at $x=0$. A front between the values $+U$ and $-U$ has the profile $-U\tanh\big(\frac{Ux}{2\nu}\big)$: its thickness is of the order of $\delta = \frac{2\nu}{U}\approx6\cdot10^{-3}$, and its slope at the center is $\frac{U^2}{2\nu}\approx157$, for $U=1$.

A front of width $0.3\,\%$ of the domain, with a slope of about 150: this is what makes the problem hard for a neural network.

In [ ]:
NU = 0.01 / np.pi          # viscosity
X_MIN, X_MAX = -1.0, 1.0   # space interval
T_MAX = 0.99               # final time


def u0(x):
    """initial condition"""
    return -np.sin(np.pi * x)

### 1.2 The exact solution (Cole–Hopf) -- for the initial condition $$u_0(x) = -\sin(\pi x)$$

The change of variables $u = -2\nu\,\partial_x\ln\varphi$ turns the Burgers equation into the heat equation $\partial_t\varphi = \nu\,\partial_{xx}\varphi$ (substitute to check). The initial condition becomes $\varphi(x,0) = \exp\big(-\frac{1}{2\nu}\int_0^x u_0(s)\,ds\big)$, proportional to $f(x) = \exp\big(-\frac{\cos(\pi x)}{2\pi\nu}\big)$, and the heat equation is solved by a convolution with the heat kernel $\frac{1}{\sqrt{4\pi\nu t}}\,e^{-\eta^2/(4\nu t)}$. Since $f'(y) = \frac{\sin(\pi y)}{2\nu}f(y)$, this gives
$$u(x,t) = -\frac{\displaystyle\int_{\mathbb R}\sin\big(\pi(x-\eta)\big)\,f(x-\eta)\,e^{-\eta^2/(4\nu t)}\,d\eta}{\displaystyle\int_{\mathbb R} f(x-\eta)\,e^{-\eta^2/(4\nu t)}\,d\eta}.$$
This formula is **exact**. It gives the solution on $\mathbb R$ with the initial condition $-\sin(\pi x)$ on $\mathbb R$. This initial condition is odd and 2-periodic; since $-u(-x,t)$ and $u(x+2,t)$ also solve the Burgers equation with the same initial condition, by uniqueness the solution is odd and 2-periodic too. Therefore $u(\pm1,t)=-u(\mp1,t)=-u(\pm1,t)$, i.e. $u(\pm1,t)=0$: it is also the solution of our problem on $[-1,1]$.



In [ ]:
def burgers_exact(x, t, nz=2001, L=12.0, chunk=1000):
    """Exact solution u(x, t) and its derivative u_x(x, t) by the Cole-Hopf formula.

    x, t: arrays of the same shape (or broadcastable). The two integrals are computed with the
    trapezoidal rule in z = eta / sqrt(4 nu t) on [-L, L], with a log-sum-exp to avoid overflow.
    """
    x, t = np.broadcast_arrays(np.asarray(x, dtype=float), np.asarray(t, dtype=float))
    shape, x, t = x.shape, x.ravel(), t.ravel()
    z = np.linspace(-L, L, nz)
    u, ux = np.empty(x.size), np.empty(x.size)
    for i in range(0, x.size, chunk):                        # by chunks, to limit the memory
        xi, ti = x[i:i + chunk, None], t[i:i + chunk, None]
        y = xi - np.sqrt(4 * NU * ti) * z                    # y = x - eta
        E = -np.cos(np.pi * y) / (2 * np.pi * NU) - z ** 2   # log of the integrand (without the sine)
        w = np.exp(E - E.max(axis=1, keepdims=True))         # log-sum-exp: the largest term is 1
        S, C = np.sin(np.pi * y), np.cos(np.pi * y)
        num, den = (S * w).sum(1), w.sum(1)
        dw = w * S / (2 * NU)                                # d w / d x
        dnum, dden = (np.pi * C * w + S * dw).sum(1), dw.sum(1)
        u[i:i + chunk] = -num / den
        ux[i:i + chunk] = -(dnum * den - num * dden) / den ** 2
    return u.reshape(shape), ux.reshape(shape)


### 1.3 Data generation

Two functions generate exact data for any number of points: on a regular grid (the file `burgers.mat` in the repository of Raissi *et al.*, is the $256\times100$ grid), or at random points, possibly with noise (useful for inverse problems). For the forward problem, the PINN does **not** use any value of $u$ inside the domain: these data only serve to evaluate it.

In [ ]:
def make_grid_data(nx=256, nt=100):
    """Exact solution on a regular grid: x (nx,), t (nt,), U (nx, nt) with U[i, k] = u(x_i, t_k)."""
    x = np.linspace(X_MIN, X_MAX, nx)
    t = np.linspace(0, T_MAX, nt)
    U = burgers_exact(x[:, None], t[None, :])[0]
    return x, t, U


def make_scattered_data(n=2000, noise=0.0, seed=0):
    """n random points (x, t) in the domain, with the exact u (plus a Gaussian noise of relative level `noise`)."""
    rng = np.random.default_rng(seed)
    xt = np.c_[rng.uniform(X_MIN, X_MAX, n), rng.uniform(0, T_MAX, n)]
    u = burgers_exact(xt[:, 0], xt[:, 1])[0]
    return xt, u + noise * np.std(u) * rng.standard_normal(n)

In [ ]:
# Data generation
x, t, U = make_grid_data(nx=256, nt=100)                   # the grid of the classical burgers.mat file
print('x:', x.shape, ' t:', t.shape, ' U:', U.shape, f'  grid step dx = {x[1] - x[0]:.2e}')


## Illustration of the test-case, data

In [ ]:
# ---- Illustration global solution
# list of time steps
t_list = [0, 25, 50, 75, 99]

# Plot
fig, axs = plt.subplots(1, 2, figsize=(15, 4.3), gridspec_kw={'width_ratios': [1.25, 1]})
im = axs[0].imshow(U, extent=[0, T_MAX, X_MIN, X_MAX], origin='lower', aspect='auto', cmap='coolwarm', vmin=-1, vmax=1)
axs[0].axvline(1 / np.pi, color='k', ls='--', lw=1); axs[0].text(1 / np.pi + 0.01, 0.85, r'$t^*=1/\pi$')
axs[0].set_xlabel('$t$'); axs[0].set_ylabel('$x$'); axs[0].set_title('exact solution $u(x,t)$'); axs[0].grid(False)
fig.colorbar(im, ax=axs[0], pad=0.02)
for k, c in zip(t_list, plt.cm.viridis(np.linspace(0, 0.9, len(t_list)))):
    axs[1].plot(x, U[:, k], color=c, lw=2, label=f'$t = {t[k]:.2f}$')
axs[1].set_xlabel('$x$'); axs[1].set_ylabel('$u$'); axs[1].set_title('snapshots'); axs[1].legend(fontsize=9)
plt.tight_layout(); plt.show()



In [ ]:
# the front: zoom, and its slope |u_x(0, t)| along time
xf = np.linspace(-0.03, 0.03, 601)
ts = np.linspace(0.01, T_MAX, 99)
slope = np.abs(burgers_exact(np.zeros_like(ts), ts)[1])
fig, axs = plt.subplots(1, 2, figsize=(15, 3.8))
for tk, c in zip([0.25, 0.5, 0.75], ['tab:blue', 'tab:orange', 'tab:green']):
    axs[0].plot(xf, burgers_exact(xf, tk)[0], color=c, lw=2, label=f'$t = {tk}$')
    k = np.argmin(np.abs(t - tk)); sel = np.abs(x) < 0.03
    axs[0].plot(x[sel], U[sel, k], 'o', color=c, ms=5)
axs[0].set_xlabel('$x$'); axs[0].set_title('zoom on the front (dots: the 256-point grid)'); axs[0].legend(fontsize=9)
axs[1].semilogy(ts, slope, color='k', lw=2); axs[1].axvline(1 / np.pi, color='k', ls='--', lw=1)
axs[1].axhline(1 / (2 * NU), color='tab:red', ls=':', lw=1.5, label=r'$U^2/2\nu$ with $U=1$')
axs[1].set_xlabel('$t$'); axs[1].set_title(r'slope of the front $|u_x(0,t)|$'); axs[1].legend()
plt.tight_layout(); plt.show()
k_max = np.argmax(slope)
print(f'steepest slope {slope[k_max]:.0f} at t = {ts[k_max]:.2f}; slope at t = 0.5: {np.abs(burgers_exact(0.0, 0.5)[1]):.0f}')

The front appears around $t^*=1/\pi$; its slope reaches 152 at $t\approx0.51$, then decreases slowly as the amplitude $U$ on each side decreases. On the 256-point grid ($\Delta x = 7.8\cdot10^{-3}$, larger than the width of the front) the front is described by one or two grid values: to evaluate a model **precisely**, we will use a finer grid (Section 3.3).

### 1.4 The training points

The PINN needs three sets of points:
* **residual (collocation) points** $(x_i,t_i)$ inside the domain, where the residual of the PDE is minimized; Indeed, we want
$$\left(\partial_t u + u\,\partial_x u - \nu\,\partial_{xx} u\right)(x_i,t_i) \approx 0$$
* **initial points** $(x_j,0)$, where 
$\left(u_\theta-u_0\right)(x_i,t_i) \approx 0$
* **boundary points** $(\pm1,t_k)$, where $u_\theta$ must vanish: 
$$u_\theta(-1,t_k) \approx 0,\quad  u_\theta(1,t_k) \approx 0$$

Collocation and boundary points: drawn by **Latin hypercube sampling** (LHS) or randomly.


In [ ]:
def lhs(n, dim, rng):
    """Latin hypercube sample of n points in [0, 1]^dim: in each dimension, one point in each of the n bins."""
    bins = np.stack([rng.permutation(n) for _ in range(dim)], axis=1)
    return (bins + rng.random((n, dim))) / n


def sample_points(n_r=5000, n_0=50, n_b=50, method='lhs', seed=0):
    """Training points: n_r residual (collocation) points inside the domain, n_0 initial points, n_b boundary times."""
    rng = np.random.default_rng(seed)
    s = lhs(n_r, 2, rng) if method == 'lhs' else rng.random((n_r, 2))
    xt_r = np.c_[X_MIN + (X_MAX - X_MIN) * s[:, 0], T_MAX * s[:, 1]]
    x_0 = np.linspace(X_MIN, X_MAX, n_0)
    t_b = T_MAX * lhs(n_b, 1, rng)[:, 0]
    xt_b = np.r_[np.c_[np.full(n_b, X_MIN), t_b], np.c_[np.full(n_b, X_MAX), t_b]]
    return {'r': to_torch(xt_r),                                   # (n_r, 2)  residual points
            'x0': to_torch(np.c_[x_0, np.zeros(n_0)]),             # (n_0, 2)  initial points (x, 0)
            'u0': to_torch(u0(x_0)[:, None]),                      # (n_0, 1)  u0(x)
            'b': to_torch(xt_b)}                                   # (2 n_b, 2) boundary points (-1, t), (1, t)

In [ ]:
## Data generation -- Choose settings
SEED = 1                            # Reproducibility
N_R, N_0, N_B = 5000, 50, 50        # residual points, initial points, boundary times (on each side)
pts = sample_points(N_R, N_0, N_B, seed=SEED)


In [ ]:
# Visualization
print({k: tuple(v.shape) for k, v in pts.items()})

P, P0, PB = to_numpy(pts['r']), to_numpy(pts['x0']), to_numpy(pts['b'])
plt.figure(figsize=(11, 4.3))
plt.imshow(U, extent=[0, T_MAX, X_MIN, X_MAX], origin='lower', aspect='auto', cmap='coolwarm', vmin=-1, vmax=1, alpha=0.55)
plt.scatter(P[:, 1], P[:, 0], s=0.6, c='k', label=f'{N_R} residual points (LHS)')
plt.scatter(P0[:, 1], P0[:, 0], s=22, marker='d', c='tab:green', label=f'{N_0} initial points')
plt.scatter(PB[:, 1], PB[:, 0], s=16, marker='s', c='tab:purple', label=f'2 x {N_B} boundary points')
plt.xlabel('$t$'); plt.ylabel('$x$'); plt.title('training points'); plt.grid(False)
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=9); plt.tight_layout(); plt.show()

in_front = np.sum((np.abs(P[:, 0]) < 2 * NU) & (P[:, 1] > 1 / np.pi))
print(f'residual points inside the front (|x| < 2 nu, t > t*): {in_front} out of {N_R}')

Only 17 of the 5000 residual points lie inside the front, where the solution changes the most. Keep this number in mind for the exercises.

## 2. Derivatives by automatic differentiation

### 2.1 The simplest model: one neuron

A PINN needs the derivatives of the network output with respect to its **inputs**. PyTorch computes them by automatic differentiation. Below, an analytical example: $u(x) = \tanh(wx+b)$, with $u' = w\,(1-u^2)$ and $u'' = -2w^2\,u\,(1-u^2)$.

In [ ]:
# Initialization
w = torch.tensor(2.0, dtype=torch.float64, requires_grad=True)          # the two parameters of the model
b = torch.tensor(0.5, dtype=torch.float64, requires_grad=True)
x = torch.linspace(-2, 2, 9, dtype=torch.float64).reshape(-1, 1).requires_grad_(True)   # a batch of 9 inputs, (9, 1)
u = torch.tanh(w * x + b)                                                 # (9, 1)

# Pytorch du/dx, d2u/dx2
u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]        # (9, 1)
u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]   # (9, 1)

# Analytical derivatives
u_x_an  = w*(1-u**2)
u_xx_an = -2*w**2*u*(1-u**2)

# Check
print('max error on u_x  (analytical vs torch):', float((u_x - u_x_an).abs().max()))
print('max error on u_xx:(analytical vs torch)', float((u_xx - u_xx_an).abs().max()))


### 2.2 Two inputs $u(x,t)$: $u_x, u_t, u_{xt}, u_{xx}$ derivatives

Now the input is a tensor `xt` of shape $(N,2)$, with columns $x$ and $t$. The gradient of $u$ with respect to `xt` has shape $(N,2)$: its columns are $u_x$ and $u_t$. To get $u_{xx}$, differentiate **the column $u_x$** and keep the first column of the result. We test it on one neuron with two inputs, $u = \tanh(ax+ct+b)$, for which $u_x = a(1-u^2)$, $u_t = c(1-u^2)$, $u_{xx} = -2a^2u(1-u^2)$ and $u_{xt} = -2ac\,u(1-u^2)$.

In [ ]:
# Simple model
a, c, b = 2.0, -1.5, 0.3
xt = torch.rand(100, 2, dtype=torch.float64).requires_grad_(True)    # (N,2) Matrix. First column is x, second is t
u  = torch.tanh(a*xt[:,0:1] + c*xt[:,1:2] + b)                       # (N,1)
print(u.shape)

In [ ]:
# -- Pytorch Derivatives
# First order
du = torch.autograd.grad(u, xt, torch.ones_like(u), create_graph=True)[0]        # (N,2) first column is u_x, second is u_t
u_x, u_t = du[:, 0:1], du[:, 1:2]

# Second order: u_xx and u_xt, starting from u_x then
d2u_x = torch.autograd.grad(u_x, xt, torch.ones_like(u_x), create_graph=True)[0] # (N,2) first column is u_xx, second is u_xt
u_xx, u_xt = d2u_x[:, 0:1], d2u_x[:, 1:2]

# -- Analytical Derivatives
s = u * (1 - u**2)
u_x_an  = a * (1 - u**2)
u_t_an  = c * (1 - u**2)
u_xx_an = - 2 * a**2 * s
u_xt_an = - 2 * a * c * s

# -- Check
print(f"u_x error: { (u_x - u_x_an).abs().max().item():.2e} | u_t error: { (u_t - u_t_an).abs().max().item():.2e}")
print(f"u_xx error: {(u_xx - u_xx_an).abs().max().item():.2e} | u_xt error: {(u_xt - u_xt_an).abs().max().item():.2e}")




## 3. PINN for Burgers equation
We start by defining a model architecture, using MLP + Fourier Features.
Then the data generation, and the loss function will be defined. Finally, the training of the model will be done.



### 3.1 The network: Fourier Features on $(x,t)$ then MLP -- use of previous classes
The input $v = (x, t)$ is first mapped to

$$\gamma(v) = \left[\cos(2\pi B v),\ \sin(2\pi B v)\right] \in \mathbb{R}^{2m}$$

where $B$ is an $m \times 2$ matrix with entries drawn from $\mathcal N(0, \sigma^2)$: $m$ frequencies (`n_freq`), so $2m$ features (cos and sin). $\sigma$ sets their size: small $\sigma$, smooth fit; large $\sigma$, fine details... and noise between the training pixels (overfitting).

$B$ is an `nn.Parameter`: fixed (`trainable=False`, `requires_grad=False`) or learned with the other weights (`trainable=True`).

In [ ]:
# Personalized class "module" with nice print/show
class modulePerso(nn.Module):
    def __init__(self):
      super().__init__()

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def __repr__(self):
        return f"{super().__repr__()}\n--> {self.n_params():,} parameters "

    # This method is for illustration purpose
    def show(self, max_nodes=8, path=None):
        mods = list(self.net)
        lin = [i for i, m in enumerate(mods) if isinstance(m, nn.Linear)]
        L = [mods[lin[0]].in_features] + [mods[i].out_features for i in lin]
        acts = [""] + [type(mods[i + 1]).__name__
                       if i + 1 < len(mods) and not isinstance(mods[i + 1], nn.Linear) else ""
                       for i in lin]
        n = len(L)

        def slots(k):  # neurones dessinés, None = "⋮"
            if k <= max_nodes:
                return list(range(k))
            t = max_nodes // 2
            return list(range(t)) + [None] + list(range(k - (max_nodes - t - 1), k))

        cols = [[(i, (len(s) - 1) / 2 - j, v) for j, v in enumerate(s)]
                for i, s in enumerate(map(slots, L))]

        fig, ax = plt.subplots(figsize=(max(6, 2.2 * n), 5.5))
        for a, b in zip(cols, cols[1:]):
            for xa, ya, va in a:
                for xb, yb, vb in b:
                    if va is not None and vb is not None:
                        ax.plot([xa, xb], [ya, yb], c="grey", lw=.5, alpha=.35, zorder=1)

        colors = ["#4CAF50"] + ["#2196F3"] * (n - 2) + ["#FF9800"]
        for i, col in enumerate(cols):
            for x, y, v in col:
                if v is None:
                    ax.text(x, y, "⋮", ha="center", va="center", fontsize=20)
                else:
                    ax.scatter(x, y, s=350, color=colors[i], edgecolor="k", zorder=3)

        names = ["Input"] + [f"Hidden {i}" for i in range(1, n - 1)] + ["Output"]
        lo, hi = -max_nodes / 2 - .6, max_nodes / 2 + .3
        for i, (name, k, act) in enumerate(zip(names, L, acts)):
            ax.text(i, lo, f"{name}\n{k}" + (f"\n{act}" if act else ""), ha="center", va="top")
        for i in range(n - 1):
            ax.text(i + .5, hi, f"{L[i] * L[i + 1] + L[i + 1]:,} params",
                    ha="center", fontsize=9, color="dimgrey")

        ax.set_title(f"MLP {L} — {self.n_params():,} paramètres")
        ax.set_xlim(-.6, n - .4)
        ax.set_ylim(lo - 1.6, hi + .6)
        ax.axis("off")
        fig.tight_layout()
        if path:
            fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.show()


In [ ]:
class MLP(modulePerso):
    def __init__(self, layers, activation, last_activation=None):
        super().__init__()
        blocks = []
        for n_in, n_out in zip(layers[:-2], layers[1:-1]):      # hidden layers
            blocks += [nn.Linear(n_in, n_out), activation()]
        blocks.append(nn.Linear(layers[-2], layers[-1]))        # output layer
        if last_activation is not None:
            blocks.append(last_activation())
        self.net = nn.Sequential(*blocks)

    def forward(self, x):
        return self.net(x)


class FourierFeatures(nn.Module):
    def __init__(self, n_in, n_freq, sigma, trainable=False):
        super().__init__()
        self.B = nn.Parameter(sigma * torch.randn(n_freq, n_in), requires_grad=trainable) # (n_freq, n_in) Matrix

    def forward(self, v):                                     # (N, n_in) -> (N, 2 * n_freq)
        z = 2 * torch.pi * v @ self.B.T                       # (N, n_in) x (n_in, n_freq) -> (N, n_freq)
        return torch.cat([torch.cos(z), torch.sin(z)], dim=1) # concat (N, n_freq) and (N, n_freq) into (N, 2*n_freq)


class MLP_FF(MLP):
    """Fourier features (n_in -> 2*n_freq) followed by MLP. `layers` = layers after FF."""
    def __init__(self, n_in, n_freq, sigma, layers, activation, last_activation=None, trainable=False):
        super().__init__([2 * n_freq] + list(layers), activation, last_activation)
        self.sigma = sigma
        self.ff = FourierFeatures(n_in, n_freq, sigma, trainable)

    def forward(self, x):
        return self.net(self.ff(x))

    def _columns(self):
        cols, edges = super()._columns()
        n_freq, n_in = self.ff.B.shape
        cols[0] = [2 * n_freq, "Fourier", f"cos/sin\nσ={self.ff.sigma}", "#9C27B0"]
        cols.insert(0, [n_in, "Input", "", "#4CAF50"])
        etat = "entraînable" if self.ff.B.requires_grad else "fixe"
        edges.insert(0, f"B : {self.ff.B.numel():,} ({etat})")
        return cols, edges



In [ ]:
# Example:
"""
2 inputs (x,t), 12 frequencies, sigma=10, hidden layers of size 128x128, output of size 1,
activation = Tanh, no activation for last layer
Fourier frequencies: trainable
"""
model = MLP_FF(2, 12, 10, [128, 128, 1], activation=nn.Tanh, last_activation=None, trainable=True)
print(model)
model.show()



In [ ]:
# Frequencies, in matrix B
model.ff.B

### 3.2 The loss

$$\mathcal L(\theta)=\underbrace{\frac{1}{N_r}\sum_{i=1}^{N_r}r_\theta(x_i,t_i)^2}_{\text{PDE residual}}+\underbrace{\frac{1}{N_0}\sum_{j=1}^{N_0}\big(u_\theta(x_j,0)-u_0(x_j)\big)^2}_{\text{initial condition}}+\underbrace{\frac{1}{2N_b}\sum_{k=1}^{N_b}\Big(u_\theta(-1,t_k)^2+u_\theta(1,t_k)^2\Big)}_{\text{boundary conditions}},\qquad r_\theta = \partial_tu_\theta+u_\theta\,\partial_xu_\theta-\nu\,\partial_{xx}u_\theta.$$
Three means with equal weights: the size of each term does not depend on its number of points, and its square root is an RMS value (a residual for the first term, an error on $u$ for the two others).

In [ ]:

def derivatives(model, xt):
    """u, u_x, u_t, u_xx at the points xt (N, 2), by automatic differentiation. model: (N,2)-> (N,1)"""
    xt = xt.clone().requires_grad_(True)
    u = model(xt)
    du = torch.autograd.grad(u, xt, torch.ones_like(u), create_graph=True)[0]       # (N, 2) = (u_x, u_t)
    u_x, u_t = du[:, 0:1], du[:, 1:2]
    u_xx = torch.autograd.grad(u_x, xt, torch.ones_like(u_x), create_graph=True)[0][:, 0:1]
    return u, u_x, u_t, u_xx

def pde_residual(model, xt, nu=NU):
    """r = u_t + u u_x - nu u_xx"""
    assert nu is not None
    u, u_x, u_t, u_xx = derivatives(model, xt)
    return u_t + u * u_x - nu* u_xx

def loss_terms(model, pts, nu=NU):
    """The three terms of the loss: mean squared PDE residual, initial error, boundary error."""
    r = pde_residual(model, pts['r'], nu=nu)
    e0 = model(pts['x0']) - pts['u0']
    eb = model(pts['b'])
    return {'pde': (r ** 2).mean(), 'ic': (e0 ** 2).mean(), 'bc': (eb ** 2).mean()}

In [ ]:
# Choose model architecture
model = MLP_FF(2, 8, 1.0, [20,20,20,20,20,20,20,20, 1], activation=nn.Tanh, last_activation=None, trainable=True)
model = MLP_FF(2, 16, 1.0, [16,16,16,16,16,16,16,16,16, 1], activation=nn.Tanh, last_activation=None, trainable=True)

print(model)
model.show()

In [ ]:
terms = loss_terms(model, pts, nu=NU)
print('initial loss terms:', {k: f'{float(v.detach()):.3e}' for k, v in terms.items()})

### 3.3 Tools to evaluate a model

To judge a trained model we compare it with the exact solution, which the training never sees:
* **the relative $L^2$ error** $\varepsilon = \|u_\theta-u\|_2/\|u\|_2$ on a reference grid of $1001\times100$ points ($\Delta x=0.002$, about a third of the width of the front, so that the grid resolves the front);
* the **maximal pointwise error** and where it occurs, and the relative error at each time, $\varepsilon(t)$;
* the **mean squared residual on 20 000 fresh random points**, compared with the one on the training points: if it is much larger, the network fits the PDE at the training points but not between them.

`evaluate(model, pts)` returns all of this (and prints a one-line summary); `plot_snapshots`, `plot_errors` and `plot_history` draw the standard figures. The reference grid is computed once, by Cole–Hopf (about 10 s).

In [ ]:
X_REF, T_REF, U_REF = make_grid_data(nx=1001, nt=100)      # reference grid: dx = 0.002, never used for training
XT_REF = torch.tensor(np.stack(np.meshgrid(X_REF, T_REF, indexing='ij'), axis=-1).reshape(-1, 2), dtype=torch.float64)
XT_FRESH = torch.tensor(make_scattered_data(20000, seed=12345)[0], dtype=torch.float64)   # 20 000 fresh random points


def as_input(model, xt):
    """points (numpy or torch) -> tensor with the precision and the device of the model"""
    p = next(model.parameters())
    return torch.as_tensor(xt, dtype=p.dtype, device=p.device)


def predict(model, xt=None):
    """u_theta on the reference grid (shape (nx, nt)), or at the points xt (shape (N,))."""
    with torch.no_grad():
        if xt is None:
            return to_numpy(model(as_input(model, XT_REF))).reshape(U_REF.shape)
        return to_numpy(model(as_input(model, xt)))[:, 0]


def rel_l2(model):
    """relative L2 error on the reference grid"""
    return np.linalg.norm(predict(model) - U_REF) / np.linalg.norm(U_REF)


def evaluate(model, pts=None, name='model', verbose=True):
    """Accuracy of a trained model: errors on the reference grid, residual on fresh points (and on the training points)."""
    E = predict(model) - U_REF
    i, k = np.unravel_index(np.abs(E).argmax(), E.shape)
    res = {'eps': np.linalg.norm(E) / np.linalg.norm(U_REF),                             # relative L2 error
           'eps_t': np.linalg.norm(E, axis=0) / np.maximum(np.linalg.norm(U_REF, axis=0), 1e-12),  # per time
           'max_err': np.abs(E[i, k]), 'x_max': X_REF[i], 't_max': T_REF[k],
           'res_fresh': float(pde_residual(model, as_input(model, XT_FRESH)).pow(2).mean())}
    if pts is not None:
        res['res_train'] = float(pde_residual(model, pts['r']).pow(2).mean())
    if verbose:
        msg = (f"{name}: eps = {res['eps']:.2e} | max error {res['max_err']:.1e} at (x, t) = ({res['x_max']:+.3f}, "
               f"{res['t_max']:.2f}) | residual MSE: fresh points {res['res_fresh']:.1e}")
        print(msg + (f", training points {res['res_train']:.1e}" if pts is not None else ''))
    return res

In [ ]:
def plot_snapshots(models, times=(0.0, 0.25, 0.5, 0.75, 0.99), zoom=0.05):
    """Exact solution and predictions at a few times; second row: zoom on the front (|x| < zoom)."""
    models = models if isinstance(models, dict) else {'PINN': models}
    xf = np.linspace(-zoom, zoom, 401)
    fig, axs = plt.subplots(2, len(times), figsize=(3.6 * len(times), 6.2), sharey='row')
    for j, t in enumerate(times):
        for ax, x in zip(axs[:, j], [X_REF, xf]):
            ax.plot(x, burgers_exact(x, t)[0], color='k', lw=2.4, alpha=0.35, label='exact')
            for (name, m), ls in zip(models.items(), ['--', ':', '-.', (0, (1, 1))]):
                ax.plot(x, predict(m, np.c_[x, np.full_like(x, t)]), ls=ls, lw=1.8, label=name)
            ax.set_xlabel('$x$')
        axs[0, j].set_title(f'$t = {t:.2f}$')
    axs[0, 0].set_ylabel('$u$'); axs[1, 0].set_ylabel(f'$u$, zoom $|x|<{zoom}$'); axs[0, 0].legend(fontsize=9)
    plt.tight_layout(); plt.show()


def plot_errors(model, pts=None):
    """Map of the pointwise error |u_theta - u| (log scale), with the residual points, and relative error per time."""
    from matplotlib.colors import LogNorm
    E = np.abs(predict(model) - U_REF)
    res = evaluate(model, verbose=False)
    fig, axs = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={'width_ratios': [1.3, 1]})
    lo = E.max() * 1e-4                                            # color scale: 4 decades below the largest error
    im = axs[0].imshow(np.maximum(E, lo), extent=[0, T_MAX, X_MIN, X_MAX], origin='lower', aspect='auto',
                       cmap='magma_r', norm=LogNorm(vmin=lo, vmax=E.max()))
    if pts is not None:
        P = to_numpy(pts['r'])
        axs[0].scatter(P[:, 1], P[:, 0], s=0.2, c='tab:cyan', alpha=0.4)
    fig.colorbar(im, ax=axs[0], pad=0.02)
    axs[0].set_title(r'$|u_\theta - u|$' + (' and the residual points' if pts is not None else ''))
    axs[0].set_xlabel('$t$'); axs[0].set_ylabel('$x$'); axs[0].grid(False)
    axs[1].semilogy(T_REF, res['eps_t'], color='tab:red', lw=2)
    axs[1].axvline(1 / np.pi, color='k', ls=':', lw=1)
    axs[1].set_xlabel('$t$'); axs[1].set_title(r'relative $L^2$ error at time $t$ (dotted: $t^*=1/\pi$)')
    plt.tight_layout(); plt.show()


def plot_history(hist):
    """Loss terms and error eps along the iterations (dotted lines: change of optimizer)."""
    fig, axs = plt.subplots(1, 2, figsize=(15, 4))
    axs[0].semilogy(hist['it'], hist['loss'], color='tab:blue', lw=2.2, alpha=0.5, label='total loss')
    for k, c, lab in [('pde', '0.2', 'PDE residual'), ('ic', 'tab:green', 'initial condition'), ('bc', 'tab:purple', 'boundary')]:
        axs[0].semilogy(hist['it'], hist[k], color=c, lw=1.4, label=lab)
    axs[0].set_title('loss and its three terms'); axs[0].legend()
    axs[1].semilogy(hist['it'], hist['eps'], color='tab:red', lw=1.8)
    axs[1].set_title(r'relative $L^2$ error $\varepsilon$ (monitoring only)')
    for ax in axs:
        ax.set_xlabel('iteration')
        for it, name in hist.marks[1:]:
            ax.axvline(it, color='k', ls=':', lw=1)
    plt.tight_layout(); plt.show()

In [ ]:
res0 = evaluate(model, pts, name='untrained network')

### 3.4 Training: Adam, then L-BFGS

We use two optimizers, one after the other, on the full set of points (no mini-batches):
* **Adam** (`lr` $=10^{-3}$), for `N_ADAM` iterations;
* **L-BFGS**, a quasi-Newton method, for `N_LBFGS` iterations, starting from the Adam solution.

All the choices of the training are made in the **initialization cell** below: the training points, the network, the **weights of the loss** and the two optimizers. Run it again before any new training (otherwise the training continues from the current network).

**The loss, with weights.** The loss of Section 3.2 is written explicitly in the training loops, with one weight per term:

$$\mathcal L(\theta)= w_{\mathrm{PDE}}\,\mathrm{MSE}_{\mathrm{PDE}} + w_{\mathrm{IC}}\,\mathrm{MSE}_{\mathrm{IC}} + w_{\mathrm{BC}}\,\mathrm{MSE}_{\mathrm{BC}}.$$

The three weights `W_PDE`, `W_IC`, `W_BC` are set to 1 (the loss of Section 3.2); change them to give more importance to one term.

**Monitoring.** Every `LOG_EVERY = 100` iterations we store in `hist` the total loss, its three terms (before weighting), and the error $\varepsilon$ (monitoring only: it uses the exact solution, never seen by the training), and we redraw two live plots: the total loss, and the loss by term.

In [ ]:
# ---- Initialization: data, model, loss weights, optimizers  (run this cell again before a new training)
from tqdm.auto import tqdm                          # progress bars
from IPython.display import display                 # live plot
from torch.nn.utils import parameters_to_vector     # all the parameters in one vector (stop test of L-BFGS)

# Data: the training points (as in Section 1.4)
SEED = 1                                            # fixes the points and the initialization of the network
N_R, N_0, N_B = 5000, 50, 50                        # residual points, initial points, boundary times (on each side)
pts = sample_points(N_R, N_0, N_B, seed=SEED)

# Model (as in Section 3.2)
torch.manual_seed(SEED)
model = MLP_FF(2, 16, 1.0, [16,16,16,16,16,16,16,16,16, 1], activation=nn.Tanh, last_activation=None, trainable=True).to(DEVICE)

# Weights of the loss: loss = W_PDE * MSE_PDE + W_IC * MSE_IC + W_BC * MSE_BC
W_PDE = 1.0                                         # PDE residual
W_IC  = 1.0                                         # initial condition
W_BC  = 1.0                                         # boundary conditions

# Optimizers
N_ADAM, N_LBFGS = 2000, 5000                        # iterations of each optimizer
LOG_EVERY = 100                                     # monitoring and live plot every LOG_EVERY iterations
optimizer_adam = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer_lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0,
                                    max_iter=LOG_EVERY, max_eval=10 * LOG_EVERY,     # per call to .step()
                                    history_size=50,
                                    tolerance_grad=0.0, tolerance_change=0.0,        # no convergence test
                                    line_search_fn='strong_wolfe')

# History of the training (one value every LOG_EVERY iterations)
hist = {'it': [], 'time': [], 'loss': [], 'pde': [], 'ic': [], 'bc': [], 'eps': []}
print(model)

#### Adam

The standard loop: at each iteration, the three terms of the loss are computed on all the points, then their weighted sum; its gradient is back-propagated and the weights are updated. Every `LOG_EVERY` iterations, the values are stored in `hist` and the plots are redrawn.

In [ ]:
# ---- Training with Adam
t0 = time.time()
fig, axs = plt.subplots(1, 2, figsize=(14, 4)); live = display(fig, display_id=True); plt.close(fig)

for it in tqdm(range(N_ADAM), desc='Adam'):
    # 1. the loss: its three terms, then their weighted sum
    optimizer_adam.zero_grad()
    r = pde_residual(model, pts['r'], nu=NU)                       # r = u_t + u u_x - nu u_xx at the residual points
    loss_pde = (r ** 2).mean()                                     # PDE residual
    loss_ic = ((model(pts['x0']) - pts['u0']) ** 2).mean()         # initial condition: u(x, 0) = u0(x)
    loss_bc = (model(pts['b']) ** 2).mean()                        # boundary conditions: u(-1, t) = u(1, t) = 0
    loss = W_PDE * loss_pde + W_IC * loss_ic + W_BC * loss_bc

    # 2. gradient and update of the weights
    loss.backward()
    optimizer_adam.step()

    # 3. monitoring (values before this update) and live plot
    if it % LOG_EVERY == 0 or it == N_ADAM - 1:
        hist['it'].append(it)
        hist['time'].append(time.time() - t0)
        hist['loss'].append(loss.item())
        hist['pde'].append(loss_pde.item())
        hist['ic'].append(loss_ic.item())
        hist['bc'].append(loss_bc.item())
        hist['eps'].append(rel_l2(model))                          # error w.r.t. the exact solution (monitoring only)

        axs[0].clear(); axs[1].clear()
        axs[0].semilogy(hist['it'], hist['loss'], color='tab:blue', lw=2)
        axs[0].set_title(f'total loss = {W_PDE:g} PDE + {W_IC:g} IC + {W_BC:g} BC')
        axs[1].semilogy(hist['it'], hist['pde'], color='0.2', label='PDE residual')
        axs[1].semilogy(hist['it'], hist['ic'], color='tab:green', label='initial condition')
        axs[1].semilogy(hist['it'], hist['bc'], color='tab:purple', label='boundary')
        axs[1].set_title('loss by term (before weighting)'); axs[1].legend()
        for ax in axs:
            ax.set_xlabel('iteration')
        fig.tight_layout()
        live.update(fig)

time_adam = time.time() - t0
print(f"after Adam  : {N_ADAM} iterations, {time_adam:.0f} s | loss {hist['loss'][-1]:.2e} "
      f"(PDE {hist['pde'][-1]:.1e}, IC {hist['ic'][-1]:.1e}, BC {hist['bc'][-1]:.1e}) | eps = {rel_l2(model):.2e}")

#### L-BFGS

L-BFGS evaluates the loss and its gradient several times per iteration (line search): we give it a function, the `closure`, which computes the loss (the same weighted loss as for Adam) and its gradient. Each call to `.step(closure)` performs `max_iter = LOG_EVERY` iterations; between two calls, L-BFGS keeps its memory (the pairs $(\mathbf s_k,\mathbf q_k)$), so `N_LBFGS // LOG_EVERY` calls make `N_LBFGS` iterations. After each call, we record and redraw the plots.

| L-BFGS argument | value | meaning |
|---|---|---|
| `lr` | 1 | first trial step $\eta=1$: the quasi-Newton step, then adjusted by the line search |
| `line_search_fn` | `'strong_wolfe'` | the step must satisfy the two Wolfe conditions |
| `history_size` | 50 | number $m$ of pairs $(\mathbf s_k,\mathbf q_k)$ kept |
| `max_iter`, `max_eval` | 100, 1000 | iterations and loss evaluations per call to `.step()`; we call it repeatedly |
| `tolerance_grad`, `tolerance_change` | 0 | no convergence test: we decide the number of iterations |

The loop stops earlier if a call to `.step()` does not move the parameters: this happens when the line search finds no decrease (typically in float32, when the changes of the loss reach the rounding level).

In [ ]:
# ---- Training with L-BFGS (starts from the network trained by Adam)
def closure():
    # the loss: its three terms, then their weighted sum (as for Adam)
    optimizer_lbfgs.zero_grad()
    r = pde_residual(model, pts['r'], nu=NU)
    loss_pde = (r ** 2).mean()                                     # PDE residual
    loss_ic = ((model(pts['x0']) - pts['u0']) ** 2).mean()         # initial condition
    loss_bc = (model(pts['b']) ** 2).mean()                        # boundary conditions
    loss = W_PDE * loss_pde + W_IC * loss_ic + W_BC * loss_bc
    loss.backward()
    terms.update(pde=loss_pde.item(), ic=loss_ic.item(), bc=loss_bc.item())    # kept for the monitoring
    return loss


terms = {}                                                         # the three terms at the last call of closure()
t0 = time.time()
fig, axs = plt.subplots(1, 2, figsize=(14, 4)); live = display(fig, display_id=True); plt.close(fig)

for call in tqdm(range(N_LBFGS // LOG_EVERY), desc='L-BFGS'):
    # 1. LOG_EVERY iterations of L-BFGS
    theta_old = parameters_to_vector(model.parameters()).clone()
    optimizer_lbfgs.step(closure)

    # 2. monitoring (at the current parameters) and live plot
    loss = closure()
    n_iter = optimizer_lbfgs.state[optimizer_lbfgs.param_groups[0]['params'][0]]['n_iter']   # L-BFGS iterations done
    hist['it'].append(N_ADAM + n_iter)
    hist['time'].append(time_adam + time.time() - t0)
    hist['loss'].append(loss.item())
    hist['pde'].append(terms['pde'])
    hist['ic'].append(terms['ic'])
    hist['bc'].append(terms['bc'])
    hist['eps'].append(rel_l2(model))

    axs[0].clear(); axs[1].clear()
    axs[0].semilogy(hist['it'], hist['loss'], color='tab:blue', lw=2)
    axs[0].set_title(f'total loss = {W_PDE:g} PDE + {W_IC:g} IC + {W_BC:g} BC')
    axs[1].semilogy(hist['it'], hist['pde'], color='0.2', label='PDE residual')
    axs[1].semilogy(hist['it'], hist['ic'], color='tab:green', label='initial condition')
    axs[1].semilogy(hist['it'], hist['bc'], color='tab:purple', label='boundary')
    axs[1].set_title('loss by term (before weighting)'); axs[1].legend()
    for ax in axs:
        ax.axvline(N_ADAM, color='k', ls=':', lw=1)                # dotted line: Adam -> L-BFGS
        ax.set_xlabel('iteration')
    fig.tight_layout()
    live.update(fig)

    # 3. stop if the parameters did not move (the line search found no decrease)
    if torch.equal(theta_old, parameters_to_vector(model.parameters())):
        print(f'L-BFGS: no decrease found by the line search, stop after {n_iter} iterations')
        break

print(f"after L-BFGS: {hist['it'][-1]} iterations, {hist['time'][-1]:.0f} s | loss {hist['loss'][-1]:.2e} "
      f"(PDE {hist['pde'][-1]:.1e}, IC {hist['ic'][-1]:.1e}, BC {hist['bc'][-1]:.1e}) | eps = {hist['eps'][-1]:.2e}")

### 3.5 Results

In [ ]:
# Loss terms and error eps along the iterations (dotted line: Adam -> L-BFGS)
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
axs[0].semilogy(hist['it'], hist['loss'], color='tab:blue', lw=2.2, alpha=0.5, label='total loss')
axs[0].semilogy(hist['it'], hist['pde'], color='0.2', lw=1.4, label='PDE residual')
axs[0].semilogy(hist['it'], hist['ic'], color='tab:green', lw=1.4, label='initial condition')
axs[0].semilogy(hist['it'], hist['bc'], color='tab:purple', lw=1.4, label='boundary')
axs[0].set_title('loss and its three terms'); axs[0].legend()
axs[1].semilogy(hist['it'], hist['eps'], color='tab:red', lw=1.8)
axs[1].set_title(r'relative $L^2$ error $\varepsilon$ (monitoring only)')
for ax in axs:
    ax.set_xlabel('iteration')
    ax.axvline(N_ADAM, color='k', ls=':', lw=1)
plt.tight_layout(); plt.show()

**Reading.** After 2000 Adam iterations the error is $\varepsilon=$ $3.3\cdot10^{-1}$; L-BFGS then divides the loss by $3.4\cdot10^{4}$ and the error by 415, down to $\varepsilon=$ $7.9\cdot10^{-4}$. The loss measures the residuals of the PDE, of the initial and of the boundary conditions **at the training points**; $\varepsilon$ measures the distance to the exact solution **everywhere**. A small loss does not imply a small error (Exercise 0).

In [ ]:
res = evaluate(model, pts, name='PINN (Adam + L-BFGS)')
plot_snapshots(model)
plot_errors(model, pts)

**Reading.**
* The PINN reproduces the solution with $\varepsilon=$ $7.9\cdot10^{-4}$ (relative $L^2$ error on the $1001\times100$ grid).
* The largest error, $1.2\cdot10^{-2}$, is at $(x,t)=(+0.000,\ 0.72)$, in the front: the zoom (second row) shows where the network is least accurate.
* The error per time grows after the formation of the front ($t^*=1/\pi$).
* The mean squared residual is $1.4\cdot10^{-6}$ at the training points and $7.9\cdot10^{-5}$ on fresh points, 58 times larger: the network satisfies the PDE better at the points it was trained on than between them.

In [ ]:
print(f"{'':28s} {'value':>10s}")
for k, v in [('parameters', model.n_params()), ('iterations (Adam + L-BFGS)', hist['it'][-1]),
             ('training time (s)', f"{hist['time'][-1]:.0f}"), ('final loss', f"{hist['loss'][-1]:.2e}"),
             ('relative L2 error eps', f"{res['eps']:.2e}"), ('max pointwise error', f"{res['max_err']:.2e}"),
             ('residual MSE, training', f"{res['res_train']:.1e}"), ('residual MSE, fresh', f"{res['res_fresh']:.1e}")]:
    print(f'{k:28s} {v!s:>10s}')

## 4. Exercises: how to do better?

The PINN above is a correct, standard PINN. **Goal of the exercises: reach $\varepsilon\le10^{-4}$ reliably, with a training time of the order of 15 minutes on a CPU** (bonus: $10^{-5}$). The levers: the optimizer, the points, the network, and the loss.

Rules of the game, as in any experimental study:
* change **one thing at a time**, with the same seed (same initialization, same points);
* report the final loss, $\varepsilon$, the residual on fresh points and the training time, in a table;
* compare at a **similar cost**: wall time, or number of loss evaluations (an iteration of Adam, L-BFGS or BFGS does not cost the same);
* repeat your best candidates with **other seeds**: a difference smaller than the spread between seeds means nothing.

### Exercise 0 — Is the result reliable?
Set `SEED = 0` in Section 1.4 (it fixes both the points and the initialization), and run Sections 1.4 to 3.5 again; then `SEED = 1`. Compare $\varepsilon$ and look at the front (`plot_snapshots`). What goes wrong, and why can the loss be small while the solution is wrong? (The "Reading" cells are text: they do not change when you run the notebook again.)

### Exercise 1 — The optimizer
Start with the optimizer: as long as the loss is not well minimized, the other changes cannot be judged.
1. **Precision.** Set `DTYPE = torch.float64` at the top of the notebook and run again. Near convergence, the pairs $(\mathbf s_k,\mathbf q_k)$ of L-BFGS are differences of close numbers: how many significant digits does float32 keep?
2. **Settings.** Adam: `lr` $\in\{10^{-2},10^{-3},10^{-4}\}$, and the length of the warm-up (the number of Adam iterations before L-BFGS). L-BFGS: `history_size`, and without line search (`line_search_fn=None` with `lr=0.1`, as in the course script `a_burgers.py`).
3. **Beyond L-BFGS: self-scaled BFGS.** The network has only $d=3317$ parameters, so a **full** $d\times d$ inverse-Hessian matrix $H$ fits in memory (88 MB in float64). Implement BFGS with a strong Wolfe line search (`from torch.optim.lbfgs import _strong_wolfe`) and the self-scaling of Al-Baali, used in recent PINN studies (Urbán *et al.*, J. Comput. Phys. 2025; Kiyani *et al.*, 2025). At each iteration:
$$\mathbf p_k=-H_k\nabla\mathcal L(\theta_k),\qquad \theta_{k+1}=\theta_k+\eta_k\mathbf p_k\ (\eta_k\text{ from the line search}),\qquad \mathbf s_k=\theta_{k+1}-\theta_k,\quad \mathbf q_k=\nabla\mathcal L(\theta_{k+1})-\nabla\mathcal L(\theta_k),$$
$$H_k\leftarrow\max(1,b_k)\,H_k,\quad b_k=\frac{-\eta_k\,\mathbf s_k^\top\nabla\mathcal L(\theta_k)}{\mathbf s_k^\top\mathbf q_k},\qquad H_{k+1}=\big(I-\rho_k\mathbf s_k\mathbf q_k^\top\big)H_k\big(I-\rho_k\mathbf q_k\mathbf s_k^\top\big)+\rho_k\mathbf s_k\mathbf s_k^\top,\quad\rho_k=\frac{1}{\mathbf q_k^\top\mathbf s_k}.$$
Here $b_k=\frac{\mathbf s_k^\top B_k\mathbf s_k}{\mathbf s_k^\top\mathbf q_k}$ with $B_k=H_k^{-1}$ (the second form above uses $\mathbf s_k=-\eta_kH_k\nabla\mathcal L(\theta_k)$ and avoids inverting $H_k$): when $b_k>1$, $H_k$ is too small along $\mathbf s_k$, and it is enlarged before the update (Al-Baali, 1998).

*Hints.* Work with the flattened vector of all the parameters ($\theta\in\mathbb R^d$). `_strong_wolfe(obj_func, x, t, d, f, g, gtd)` searches along the direction `d` from `x`, starting with the step `t`; `obj_func(x, t, d)` must return the loss and the flattened gradient at `x + t*d`, and `gtd` is $\nabla\mathcal L^\top\mathbf d$; it returns the new loss, the new gradient, the step and the number of evaluations (this is a private function of PyTorch). At the first iteration, start from $H_0=I$, then replace it by $\frac{\mathbf s_0^\top\mathbf q_0}{\mathbf q_0^\top\mathbf q_0}I$ instead of the scaling. Skip the update if $\mathbf s_k^\top\mathbf q_k\le0$. Use it after a short Adam warm-up (1000 iterations), in float64. Does it also fix Exercise 0?

### Exercise 2 — The points
1. **How many points?** Compare the mean squared residual at the training points and on fresh points, for the course PINN and for your best optimizer. Then train the course PINN with $N_r\in\{1000, 10000\}$. What happens with too few points?
2. **Resampling.** Replace the residual points by new random points every $K$ iterations (e.g. $K=500$). Why should the gap between the training residual and the fresh residual close? (With L-BFGS, create a new optimizer after each resampling: its stored pairs belong to the old loss.)
3. **Adaptive sampling (RAD).** Draw the new points among many uniform candidates (e.g. $10^5$), with a probability $p(\mathbf z)\propto\frac{|r_\theta(\mathbf z)|^k}{\mathbb E|r_\theta|^k}+c$ (Wu *et al.*, 2023; take $k=1$, $c=1$). Plot the points: where do they go? Compare with uniform resampling.

### Exercise 3 — The network
1. **Fourier features.** Compare with no Fourier features (`n_fourier=0`), and with a larger scale $\sigma$ (e.g. 5). Is the network of the course a good choice?
2. **Width and depth.** Compare networks (width $\times$ depth) with a similar number of parameters: $20\times8$ (3317), $26\times6$ (3995), $32\times4$ (3761).
3. **Hard constraints.** Write $u_\theta(x,t) = u_0(x) + t\,(1-x^2)\,N_\theta(x,t)$, where $N_\theta$ is the network. Check that the initial and boundary conditions now hold exactly, whatever $\theta$. Which terms of the loss remain? Does it help?
4. **Symmetry.** Prove the symmetry used in Section 1.2: the exact solution is odd in $x$, $u(-x,t)=-u(x,t)$. Build a network that is odd by construction, $u_\theta(x,t)=\frac12\big(N_\theta(x,t)-N_\theta(-x,t)\big)$. What is $u_\theta(0,t)$, whatever $\theta$? Where is the front?

### Exercise 4 — The loss
Plot the share of each term of the loss during the training. At the end, compare $\sqrt{\text{MSE}_{\mathrm{IC}}}$ and $\sqrt{\text{MSE}_{\mathrm{BC}}}$ (RMS errors on $u$ at the initial and boundary points) with the RMS error of $u$ in the domain. Would weights $\lambda_{\mathrm{IC}},\lambda_{\mathrm{BC}}\neq1$ in front of these terms help? (change `W_IC`, `W_BC` in the initialization cell of Section 3.4.)

### Exercise 5 — Your best PINN
Combine what works, and report your best configuration over 3 seeds:

| configuration | iterations | time | final loss | $\varepsilon$ | residual MSE, fresh |
|---|---|---|---|---|---|
| this notebook (Adam + L-BFGS, float32) | | | | | |
| ... | | | | | |